In [2]:
import pandas as pd
import numpy as np
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

In [3]:
df = pd.read_csv(r"C:\Users\User\Desktop\water\clean_water_stats.csv")

In [14]:
df

,Unnamed: 0,id,date_insert,year,month,day,hour,sensor_id,lat,lng,wtempV_°C,conductivity_of_solution_ mS_cm,oxidation_reduction_potential_ mV,total_suspended_solids_mg_L,turbidity_NTU,dissolved_oxygen_concentration _ppm,dissolved_oxygen_concentration_mg_L,oxygen_saturation_%,salinity_ppt,dissolved_solids _ppm
0,2,3.0,2025-07-03 15:00:00,2025,7,3,15,976,38.310409,21.784038,24.96,56828.55,-170.59,63.73,80.74,7.91,7.91,95.62,37.99,37057.90
1,11,12.0,2025-07-03 16:00:00,2025,7,3,16,976,38.310409,21.784038,25.16,53948.80,-171.19,389.40,261.58,7.88,7.88,95.34,35.70,35180.02
2,20,21.0,2025-07-03 17:00:00,2025,7,3,17,976,38.310409,21.784038,24.70,56596.52,-170.08,432.16,175.68,8.01,8.01,95.85,37.81,36906.59
3,29,30.0,2025-07-03 18:00:00,2025,7,3,18,976,38.310409,21.784038,24.63,56950.69,-170.49,76.01,212.74,8.04,8.04,96.25,38.09,37137.55
4,38,39.0,2025-07-03 19:00:00,2025,7,3,19,976,38.310409,21.784038,24.66,56709.95,-171.16,135.79,237.97,7.99,7.99,96.00,37.90,36980.56
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
17952,28337,28338.0,2026-01-07 11:00:00,2026,1,7,11,984,38.183667,21.678139,15.89,55877.27,301.48,209.88,138.56,0.95,0.95,44.55,37.23,36437.57
17953,28342,28343.0,2026-01-07 12:00:00,2026,1,7,12,984,38.183667,21.678139,15.89,55671.35,302.35,209.88,138.56,0.31,0.31,44.55,37.07,36303.29
17954,28347,28348.0,2026-01-07 13:00:00,2026,1,7,13,984,38.183667,21.678139,15.89,56022.63,302.21,209.88,138.56,0.01,0.01,44.55,37.35,36532.36
17955,28352,28353.0,2026-01-07 14:00:00,2026,1,7,14,984,38.183667,21.678139,15.89,56145.40,301.91,209.88,138.56,0.73,0.73,44.55,37.45,36612.41


In [5]:
# 1. Define your core clean features
FEATURE_COLS = [
    'wtempV_°C', 
    'conductivity_of_solution_ mS_cm', 
    'oxidation_reduction_potential_ mV', 
    'turbidity_NTU', 
    'oxygen_saturation_%'
]

# List to collect processed dataframes for each sensor
processed_sensor_dfs = []

# Group by the unique station ID (976, 984, etc.)
for sensor_id, sensor_df in df.groupby('sensor_id'):
    
    # Work on a copy of this specific station's data
    s_df = sensor_df.copy()
    
    # Step A: Scale features ONLY using this station's local mean and std
    scaler = StandardScaler()
    X_sensor_scaled = scaler.fit_transform(s_df[FEATURE_COLS])
    
    # Step B: Fit PCA specifically on this station's local dynamics
    # We choose 3 components to retain local variance cleanly
    pca = PCA(n_components=3)
    X_pca = pca.fit_transform(X_sensor_scaled)
    
    # Step C: Assign local PC values back to this station's rows
    s_df['PC1'] = X_pca[:, 0]
    s_df['PC2'] = X_pca[:, 1]
    s_df['PC3'] = X_pca[:, 2]
    
    # Store local variance explained for verification
    var_explained = pca.explained_variance_ratio_
    print(f"Sensor ID {sensor_id} PCA Variance: PC1={var_explained[0]:.2%}, PC2={var_explained[1]:.2%}, PC3={var_explained[2]:.2%} | Total={sum(var_explained):.2%}")
    
    processed_sensor_dfs.append(s_df)

# Recombine all 9 sensors back into a single main DataFrame
df_pca_by_sensor = pd.concat(processed_sensor_dfs, axis=0).sort_index()

Sensor ID 976 PCA Variance: PC1=37.23%, PC2=22.61%, PC3=16.76% | Total=76.61%
Sensor ID 977 PCA Variance: PC1=43.72%, PC2=34.86%, PC3=9.30% | Total=87.87%
Sensor ID 978 PCA Variance: PC1=48.28%, PC2=21.20%, PC3=18.15% | Total=87.63%
Sensor ID 979 PCA Variance: PC1=41.80%, PC2=24.68%, PC3=16.26% | Total=82.73%
Sensor ID 980 PCA Variance: PC1=39.83%, PC2=23.85%, PC3=18.02% | Total=81.70%
Sensor ID 981 PCA Variance: PC1=41.11%, PC2=25.12%, PC3=20.72% | Total=86.95%
Sensor ID 982 PCA Variance: PC1=43.21%, PC2=25.28%, PC3=15.55% | Total=84.04%
Sensor ID 983 PCA Variance: PC1=38.23%, PC2=30.13%, PC3=17.24% | Total=85.60%
Sensor ID 984 PCA Variance: PC1=44.11%, PC2=27.11%, PC3=14.00% | Total=85.23%


In [6]:
sensor_loadings = {}
sensor_means = {}

for sensor_id, sensor_df in df.groupby('sensor_id'):
    s_df = sensor_df.copy()
    
    # 1. Scale locally & store the scaler means/stds
    scaler = StandardScaler()
    X_sensor_scaled = scaler.fit_transform(s_df[FEATURE_COLS])
    
    sensor_means[sensor_id] = pd.DataFrame({
        'mean': scaler.mean_,
        'std': scaler.scale_
    }, index=FEATURE_COLS)
    
    # 2. Fit local PCA
    pca = PCA(n_components=3)
    pca.fit(X_sensor_scaled)
    
    # 3. Extract Loadings (components_ transposed: rows=features, cols=PCs)
    loadings_df = pd.DataFrame(
        pca.components_.T, 
        columns=['PC1', 'PC2', 'PC3'], 
        index=FEATURE_COLS
    )
    
    sensor_loadings[sensor_id] = loadings_df

# --- PRINT FUNCTION FOR INSPECTION ---
def inspect_sensor_pca(sensor_id):
    print(f"\n=================== SENSOR ID: {sensor_id} ===================")
    print("\n--- 1. Original Feature Means & Stds (Local Baseline) ---")
    print(sensor_means[sensor_id].round(2))
    
    print("\n--- 2. Feature Loadings (Weights) ---")
    print(sensor_loadings[sensor_id].round(4))

In [7]:
for i in list(df['sensor_id'].unique()):
    inspect_sensor_pca(i)


=================== SENSOR ID: 976 ===================

--- 1. Original Feature Means & Stds (Local Baseline) ---
                                       mean     std
wtempV_°C                             24.32    0.76
conductivity_of_solution_ mS_cm    56969.70  321.56
oxidation_reduction_potential_ mV   -117.84   31.62
turbidity_NTU                        120.72   82.13
oxygen_saturation_%                   89.26    5.80

--- 2. Feature Loadings (Weights) ---
                                      PC1     PC2     PC3
wtempV_°C                          0.4478  0.3690 -0.0481
conductivity_of_solution_ mS_cm    0.1237  0.7629 -0.4545
oxidation_reduction_potential_ mV  0.4360  0.2027  0.7629
turbidity_NTU                      0.5981 -0.2295 -0.0573
oxygen_saturation_%               -0.4861  0.4336  0.4537

=================== SENSOR ID: 977 ===================

--- 1. Original Feature Means & Stds (Local Baseline) ---
                                       mean      std
wtempV_°C         

In [8]:
pc1_summary = pd.DataFrame({
    f'Sensor_{s_id}': sensor_loadings[s_id]['PC1'] 
    for s_id in sensor_loadings.keys()
})

print("--- PC1 Feature Loadings Across All 9 Sensors ---")
print(pc1_summary.round(3))

--- PC1 Feature Loadings Across All 9 Sensors ---
                                   Sensor_976  Sensor_977  Sensor_978  \
wtempV_°C                               0.448       0.623       0.584   
conductivity_of_solution_ mS_cm         0.124       0.468       0.594   
oxidation_reduction_potential_ mV       0.436      -0.565      -0.050   
turbidity_NTU                           0.598       0.241      -0.483   
oxygen_saturation_%                    -0.486       0.123       0.265   

                                   Sensor_979  Sensor_980  Sensor_981  \
wtempV_°C                               0.574       0.567       0.241   
conductivity_of_solution_ mS_cm         0.237      -0.566       0.525   
oxidation_reduction_potential_ mV       0.607      -0.491      -0.111   
turbidity_NTU                           0.156       0.155       0.545   
oxygen_saturation_%                     0.471      -0.305       0.597   

                                   Sensor_982  Sensor_983  Sensor_984  


In [30]:
loading_summary = []

for sensor_id, df_load in sensor_loadings.items():
    pc1_top = df_load['PC1'].abs().idxmax()
    pc2_top = df_load['PC2'].abs().idxmax()
    pc3_top = df_load['PC3'].abs().idxmax()
    
    loading_summary.append({
        'sensor_id': sensor_id,
        'PC1_Driver': f"{pc1_top} ({df_load.loc[pc1_top, 'PC1']:.2f})",
        'PC2_Driver': f"{pc2_top} ({df_load.loc[pc2_top, 'PC2']:.2f})",
        'PC3_Driver': f"{pc3_top} ({df_load.loc[pc3_top, 'PC3']:.2f})"
    })

pd.DataFrame(loading_summary).set_index('sensor_id')

,PC1_Driver,PC2_Driver,PC3_Driver
sensor_id,,,
976,turbidity_NTU (0.60),conductivity_of_solution_ mS_cm (0.76),oxidation_reduction_potential_ mV (0.76)
977,wtempV_°C (0.62),oxygen_saturation_% (0.67),oxidation_reduction_potential_ mV (0.78)
978,conductivity_of_solution_ mS_cm (0.59),oxidation_reduction_potential_ mV (0.88),oxygen_saturation_% (0.81)
979,oxidation_reduction_potential_ mV (0.61),turbidity_NTU (0.68),conductivity_of_solution_ mS_cm (0.67)
980,wtempV_°C (0.57),oxygen_saturation_% (0.60),turbidity_NTU (0.80)
981,oxygen_saturation_% (0.60),wtempV_°C (0.70),oxidation_reduction_potential_ mV (0.66)
982,conductivity_of_solution_ mS_cm (0.56),turbidity_NTU (0.80),oxygen_saturation_% (0.87)
983,oxidation_reduction_potential_ mV (0.65),wtempV_°C (0.73),oxygen_saturation_% (0.75)
984,wtempV_°C (0.61),oxidation_reduction_potential_ mV (0.68),oxidation_reduction_potential_ mV (0.71)


In [17]:
df_pca_by_sensor.to_csv(r"C:\Users\User\Desktop\water\water_pca.csv")